# BioLoc step-by-step pipeline

This notebook loads one synchronized segment using BioLoc's standard loader, runs every pipeline stage explicitly, and keeps the important intermediate results available for inspection and plotting. Edit only the input cell before running all cells.

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt

from bioloc import (
    BirdNETClassifier,
    PerchClassifier,
    PipelineConfig,
    associate_tdoas,
    classify_reconstructed_sources,
    estimate_local_tdoas,
    extract_regions_of_interest,
    extract_tdoa_peaks,
    load_recording,
    localize_sources,
    reconstruct_sources,
)
from bioloc.plotting import (
    plot_feature_support,
    plot_local_tdoas,
    plot_localizations,
    plot_pair_histogram,
    plot_peak_timefrequency_support,
    plot_reconstructed_signal,
    plot_reconstruction_mask,
    plot_roi,
)

%matplotlib inline

## 1. Input settings

Use the same `root / recorder ID / optional date / WAV` structure documented in the README.

In [ ]:
RECORDING_ROOT = Path("recordings")
POSITIONS_CSV = RECORDING_ROOT / "positions.csv"
RECORDER_IDS = ["mic-a", "mic-b", "mic-c"]
RECORDING_DATE = "20260102"  # None if WAVs are directly in each ID folder
RECORDING_TIME = "030405"    # None if each selected folder has one WAV
START_SECONDS = 0.0
DURATION_SECONDS = 5.0
CLASSIFIER = "birdnet"       # "birdnet" or "perch"
MIN_CONFIDENCE = 0.1
PERCH_MODEL = "perch_8"

## 2. Load and standardize the recording

In [ ]:
recording = load_recording(
    root_directory=RECORDING_ROOT,
    positions_csv=POSITIONS_CSV,
    recorder_ids=RECORDER_IDS,
    recording_date=RECORDING_DATE,
    recording_time=RECORDING_TIME,
    start_seconds=START_SECONDS,
    duration_seconds=DURATION_SECONDS,
)

environment_size = float(recording.metadata["map_info"]["environment_size"])
bounds = ((0.0, 0.0), (environment_size, environment_size))
config = PipelineConfig()
config = replace(
    config,
    association=replace(config.association, environment_bounds=bounds),
    localization=replace(config.localization, environment_bounds=bounds),
    classification=replace(
        config.classification,
        backend=CLASSIFIER,
        min_confidence=MIN_CONFIDENCE,
        perch_model=PERCH_MODEL,
    ),
)
recording

## 3. Region-of-interest extraction

In [ ]:
roi = extract_regions_of_interest(recording, config.roi, config.local_tdoa)
pair = next(iter(roi.masks_by_pair))
print("Available pairs:", list(roi.masks_by_pair))
print("Selected pair:", pair, "mask shape:", roi.for_pair(*pair).shape)
plot_roi(roi, pair);

## 4. Local TDOA estimation

In [ ]:
local_tdoas = estimate_local_tdoas(
    recording, roi, config.local_tdoa, config.roi
)
pair_local = local_tdoas.for_pair(*pair)
print("Local delay observations:", pair_local.delays.size)
plot_local_tdoas(local_tdoas, pair);

## 5. Histograms and peak picking

In [ ]:
peaks = extract_tdoa_peaks(local_tdoas, config.peaks)
print("Selected-pair peaks:", peaks.for_pair(*pair).peaks)
plot_pair_histogram(peaks, pair);

## 6. Feature construction and TDOA association

In [ ]:
association = associate_tdoas(
    peaks, local_tdoas, recording, config.association, config.local_tdoa
)
print("Feature vectors:", len(association.features))
print("Associated sources:", len(association.sources))
pair_features = association.features_for_pair(*pair)
if pair_features:
    plot_feature_support(association, pair, peak_index=0)
support_figures = plot_peak_timefrequency_support(
    peaks,
    local_tdoas,
    association,
    frames_per_bin=config.association.frames_per_bin,
)

## 7. Source localization

In [ ]:
localization = localize_sources(
    association, recording, config.localization, config.local_tdoa
)
print("Local positions (m):\n", localization.positions)
print("GPS positions [latitude, longitude]:\n", localization.gps_positions)
plot_localizations(recording, localization);

## 8. Source separation and reconstruction

In [ ]:
reconstruction = reconstruct_sources(
    recording,
    peaks,
    local_tdoas,
    association,
    localization,
    config.reconstruction,
    config.local_tdoa,
)
print("Reconstructed sources:", len(reconstruction.sources))
for source in reconstruction.sources:
    plot_reconstruction_mask(reconstruction, source.source_id)
    plot_reconstructed_signal(reconstruction, source.source_id)

## 9. Source classification

In [ ]:
if config.classification.backend == "birdnet":
    classifier = BirdNETClassifier(
        min_confidence=config.classification.min_confidence
    )
elif config.classification.backend == "perch":
    classifier = PerchClassifier(
        config.classification.perch_model,
        min_confidence=config.classification.min_confidence,
        hop_duration=config.classification.perch_hop_duration,
        warm_up=config.classification.perch_warm_up,
    )
else:
    raise ValueError("CLASSIFIER must be 'birdnet' or 'perch'")

classification = classify_reconstructed_sources(reconstruction, classifier)
for source in classification.sources:
    print(
        source.source_id,
        source.label,
        source.confidence,
        source.detections,
    )

All intermediate variables (`roi`, `local_tdoas`, `peaks`, `association`, `localization`, `reconstruction`, and `classification`) remain available for further inspection.